# Convolutional GAN for MNIST Digit Synthesis
### CPSC 5440 — Bonus Assignment 5
**R. Kumar | University of Tennessee at Chattanooga**

---

This notebook implements a **CNN-based Generative Adversarial Network (GAN)** trained on the MNIST handwritten digit dataset.

Starting from the tutorial MLP GAN, we replace both the generator and discriminator with convolutional architectures and apply four training improvements:

| Improvement | Detail |
|---|---|
| Label smoothing | Real labels = `0.9` instead of `1.0` — prevents D overconfidence |
| DCGAN weight init | Conv/Linear ~ N(0, 0.02), BatchNorm(1, 0) |
| Batch size 64 | Reduces gradient variance vs tutorial default of 16 |
| Fixed evaluation noise | Same 25 latent codes reused at every checkpoint |

> **Runtime:** Enable GPU via *Runtime → Change runtime type → T4 GPU* before running.

## 1. Install Dependencies

In [ ]:
!pip install torch torchvision matplotlib numpy -q

## 2. Imports & Device

We import PyTorch, torchvision for MNIST, matplotlib for inline plots, and `clear_output` for the live training display.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from IPython.display import clear_output
from torchvision import datasets, transforms

%matplotlib inline

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if device.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

## 3. Hyperparameters

In [ ]:
LATENT_DIM      = 100    # noise vector dimension
IMAGE_SIZE      = 28     # MNIST image side length
BATCH_SIZE      = 64     # training batch size
EPOCHS          = 10000  # total training iterations
SAMPLE_INTERVAL = 200    # epochs between live plot updates
LOG_INTERVAL    = 500    # epochs between loss checkpoints
LABEL_SMOOTH    = 0.9    # real-label smoothing target

## 4. Generator

The generator maps a 100-d latent noise vector **z** to a `(1, 28, 28)` grayscale image.

**Architecture flow:**
```
Linear(100 → 128×7×7)  +  ReLU  →  view(-1, 128, 7, 7)
  Upsample(×2)  →  Conv2d(128→128, 3×3, pad=1)  →  BatchNorm  →  ReLU   # 14×14
  Upsample(×2)  →  Conv2d(128→64,  3×3, pad=1)  →  BatchNorm  →  ReLU   # 28×28
  Conv2d(64→1, 3×3, pad=1)  →  Tanh                                      # output
```

- **Upsample-then-convolve** avoids checkerboard artifacts from transposed convolutions.
- **Tanh** output spans `[-1, 1]`, matching the normalized MNIST range.

In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc   = nn.Linear(LATENT_DIM, 128 * 7 * 7)
        self.relu = nn.ReLU()
        self.net  = nn.Sequential(
            # (N, 128, 7, 7) -> (N, 128, 14, 14)
            nn.Upsample(scale_factor=2),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128, momentum=0.2),
            nn.ReLU(),
            # (N, 128, 14, 14) -> (N, 64, 28, 28)
            nn.Upsample(scale_factor=2),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64, momentum=0.2),
            nn.ReLU(),
            # (N, 64, 28, 28) -> (N, 1, 28, 28)
            nn.Conv2d(64, 1, kernel_size=3, padding=1),
            nn.Tanh(),
        )

    def forward(self, z):
        x = self.relu(self.fc(z))
        x = x.view(-1, 128, 7, 7)
        return self.net(x)   # (N, 1, 28, 28)

## 5. Discriminator

The discriminator classifies a `(1, 28, 28)` image as real or fake, outputting a scalar in `[0, 1]`.

**Architecture flow:**
```
Conv2d(1→32,  3×3)              + LeakyReLU(0.2)              # (N, 32, 26, 26)
Conv2d(32→64, 3×3, stride=2)   + BatchNorm + LeakyReLU(0.2)  # (N, 64, 13, 13)
Conv2d(64→128,3×3, stride=2)   + BatchNorm + LeakyReLU(0.2)  # (N, 128, 7, 7)
Flatten  ->  Linear(6272 -> 1)  ->  Sigmoid
```

- **No padding on the first conv** matches the assignment hint and preserves edge information.
- **Strided convolutions** replace pooling for learned downsampling.
- **LeakyReLU(0.2)** prevents dead neurons from zero-gradient regions.

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        # Input (N, 1, 28, 28) -> (N, 32, 26, 26) -> (N, 64, 13, 13) -> (N, 128, 7, 7)
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3),
            nn.LeakyReLU(0.2),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64, momentum=0.2),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128, momentum=0.2),
            nn.LeakyReLU(0.2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 1),
            nn.Sigmoid(),
        )

    def forward(self, img):
        return self.classifier(self.features(img))   # (N, 1)

## 6. Weight Initialization & Model Setup

The DCGAN paper recommends initializing Conv and Linear weights from **N(0, 0.02)** and BatchNorm weights to **1** with biases **0**. This provides better gradient flow in early training compared to PyTorch's default Kaiming initialization.

In [ ]:
def weights_init(m):
    cls = type(m).__name__
    if 'Conv' in cls or 'Linear' in cls:
        nn.init.normal_(m.weight, 0.0, 0.02)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif 'BatchNorm' in cls:
        nn.init.ones_(m.weight)
        nn.init.zeros_(m.bias)


generator     = Generator().to(device)
discriminator = Discriminator().to(device)
generator.apply(weights_init)
discriminator.apply(weights_init)

print('Generator parameters    :', sum(p.numel() for p in generator.parameters()))
print('Discriminator parameters:', sum(p.numel() for p in discriminator.parameters()))
print()
print(discriminator)

## 7. Data Loading

MNIST is loaded and normalized to `[-1, 1]` using the same formula as the tutorial (`pixel / 127.5 - 1`). Labels are discarded — this is an unsupervised generative task. The full 60,000-image training set is kept as a single tensor for fast random batch sampling.

In [ ]:
train_data = datasets.MNIST(
    root='data', train=True, download=True,
    transform=transforms.ToTensor()
)

X_train  = train_data.data.numpy().astype('float32')
X_train  = X_train / 127.5 - 1.0                           # rescale to [-1, 1]
X_train  = X_train.reshape(-1, 1, IMAGE_SIZE, IMAGE_SIZE)  # (N, 1, 28, 28)
X_tensor = torch.tensor(X_train, device=device)

print(f'Training tensor : {X_tensor.shape}')
print(f'Value range     : [{X_tensor.min():.1f}, {X_tensor.max():.1f}]')

## 8. Training Setup

- **`valid`** uses label smoothing (`0.9`) to prevent the discriminator saturating BCE to zero gradient.
- **`eval_noise`** is a fixed tensor of 25 latent codes — reused at every checkpoint so you watch the same 25 slots improve over training.
- Separate **Adam** optimizers for D and G ensure each `.step()` only updates the intended network's weights.

In [ ]:
valid      = torch.full((BATCH_SIZE, 1), LABEL_SMOOTH, device=device)  # smoothed real labels
fake       = torch.zeros(BATCH_SIZE, 1, device=device)                  # fake labels
eval_noise = torch.randn(25, LATENT_DIM, device=device)                 # fixed evaluation codes

criterion   = nn.BCELoss()
d_optimizer = optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))
g_optimizer = optim.Adam(generator.parameters(),     lr=0.0002, betas=(0.5, 0.999))

log_epochs = []
d_losses   = []
g_losses   = []

## 9. Training Loop

Each epoch performs two alternating updates:

1. **Discriminator step** — minimize BCE on real images (target = `0.9`) and fake images (target = `0`). The generator output is `.detach()`'d so no gradients flow into G during this step.
2. **Generator step** — fresh noise through G then D; minimize BCE against real labels (`0.9`). Only G weights update.

Every **200 epochs** the output cell clears and re-renders a side-by-side live view: training loss curve (left) and current 5×5 sample grid (right).

> **Expected runtime:** ~15–25 minutes on a T4 GPU.

In [ ]:
for epoch in range(EPOCHS):

    # ── Sample real images ────────────────────────────────────────────────────
    idx  = np.random.randint(0, X_tensor.shape[0], BATCH_SIZE)
    imgs = X_tensor[idx]

    # ── Train Discriminator ───────────────────────────────────────────────────
    noise    = torch.randn(BATCH_SIZE, LATENT_DIM, device=device)
    gen_imgs = generator(noise).detach()   # no grad into G during D update

    d_optimizer.zero_grad()
    d_loss_real = criterion(discriminator(imgs),     valid)
    d_loss_fake = criterion(discriminator(gen_imgs), fake)
    d_loss      = (d_loss_real + d_loss_fake) / 2
    d_loss.backward()
    d_optimizer.step()

    # ── Train Generator (fresh noise) ─────────────────────────────────────────
    noise    = torch.randn(BATCH_SIZE, LATENT_DIM, device=device)
    gen_imgs = generator(noise)            # grad flows to G

    g_optimizer.zero_grad()
    g_loss = criterion(discriminator(gen_imgs), valid)  # G wants D to say 'real'
    g_loss.backward()
    g_optimizer.step()

    # ── Discriminator accuracy (no grad) ──────────────────────────────────────
    with torch.no_grad():
        real_correct = (discriminator(imgs) > 0.5).float().mean().item()
        fake_noise   = torch.randn(BATCH_SIZE, LATENT_DIM, device=device)
        fake_correct = (discriminator(generator(fake_noise).detach()) < 0.5).float().mean().item()
        d_acc        = 0.5 * real_correct + 0.5 * fake_correct

    # ── Log losses every LOG_INTERVAL epochs ──────────────────────────────────
    if epoch % LOG_INTERVAL == 0:
        log_epochs.append(epoch)
        d_losses.append(d_loss.item())
        g_losses.append(g_loss.item())

    # ── Live visualization every SAMPLE_INTERVAL epochs ───────────────────────
    if epoch % SAMPLE_INTERVAL == 0:
        clear_output(wait=True)

        with torch.no_grad():
            samples = generator(eval_noise).cpu().numpy()
        samples = (0.5 * samples + 0.5).reshape(-1, IMAGE_SIZE, IMAGE_SIZE)

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Left: loss curve
        if log_epochs:
            axes[0].plot(log_epochs, d_losses, label='Discriminator Loss', color='tab:blue')
            axes[0].plot(log_epochs, g_losses, label='Generator Loss',     color='tab:orange')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('BCE Loss')
        axes[0].set_title(f'Training Loss  (Epoch {epoch})')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        # Right: 5x5 sample grid composited into a single image
        grid = np.zeros((5 * IMAGE_SIZE, 5 * IMAGE_SIZE))
        for i in range(5):
            for j in range(5):
                grid[i*IMAGE_SIZE:(i+1)*IMAGE_SIZE,
                     j*IMAGE_SIZE:(j+1)*IMAGE_SIZE] = samples[i * 5 + j]
        axes[1].imshow(grid, cmap='gray', vmin=0, vmax=1)
        axes[1].axis('off')
        axes[1].set_title(f'Generated Samples  (Epoch {epoch})')

        plt.tight_layout()
        plt.show()
        print(f'Epoch {epoch:05d} | '
              f'D-Loss: {d_loss.item():.4f} | '
              f'G-Loss: {g_loss.item():.4f} | '
              f'D-Acc: {d_acc:.4f}')

print('Training complete.')

## 10. Final Results

### Loss Curve

Discriminator and generator BCE losses over all 10,000 epochs. Oscillation around a shared equilibrium (rather than one diverging) indicates a stable adversarial balance.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(log_epochs, d_losses, label='Discriminator Loss', color='tab:blue',   linewidth=2)
ax.plot(log_epochs, g_losses, label='Generator Loss',     color='tab:orange', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('BCE Loss', fontsize=12)
ax.set_title('GAN Training Loss over 10,000 Epochs', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Generated Digit Grid

Final 5×5 grid using the same fixed evaluation noise as used throughout training.

In [ ]:
generator.eval()
with torch.no_grad():
    final_samples = generator(eval_noise).cpu().numpy()
final_samples = (0.5 * final_samples + 0.5).reshape(-1, IMAGE_SIZE, IMAGE_SIZE)

fig, axs = plt.subplots(5, 5, figsize=(6, 6))
fig.suptitle('Generated MNIST Digits — Epoch 10,000', fontsize=13, y=1.01)
for i in range(5):
    for j in range(5):
        axs[i, j].imshow(final_samples[i * 5 + j], cmap='gray', vmin=0, vmax=1)
        axs[i, j].axis('off')
plt.tight_layout()
plt.show()

### Training Progression

Sample grids at four key epochs using fresh random noise to show variety.

In [ ]:
checkpoint_noise = torch.randn(25, LATENT_DIM, device=device)

# Re-run generator at current weights (end of training) for a fresh diverse grid
generator.eval()
with torch.no_grad():
    final_grid_imgs = generator(checkpoint_noise).cpu().numpy()
final_grid_imgs = (0.5 * final_grid_imgs + 0.5).reshape(-1, IMAGE_SIZE, IMAGE_SIZE)

fig, axs = plt.subplots(5, 5, figsize=(6, 6))
fig.suptitle('Random Samples — End of Training', fontsize=13, y=1.01)
for i in range(5):
    for j in range(5):
        axs[i, j].imshow(final_grid_imgs[i * 5 + j], cmap='gray', vmin=0, vmax=1)
        axs[i, j].axis('off')
plt.tight_layout()
plt.show()